In [0]:
import requests
import pandas as pd


response = requests.get("https://api.data.gov.my/weather/warning/earthquake")
data = response.json()
df = pd.DataFrame(data)
display(df)
#print(response.json())

In [0]:
print(type(df))
spark_df = spark.createDataFrame(df)   
spark_df.createOrReplaceTempView("my_df")
display(spark_df)

In [0]:

from pyspark.sql import functions as F
spark_df2 = spark_df.withColumn(
    "utcdatetime_ts",
    F.expr("try_to_timestamp(utcdatetime, \"yyyy-MM-dd'T'HH:mm:ss\")")
)

In [0]:
display(spark_df2)

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

result = (
  spark_df2.filter(F.col("utcdatetime_ts").between("2026-01-01",datetime.now()))
    .select("utcdatetime")
    .orderBy(F.col("utcdatetime").desc())
    .limit(100)
)

display(result)

In [0]:
spark_df2.createOrReplaceTempView("df2")
result = spark.sql("""
  SELECT
    *
  FROM df2
  WHERE n_distancemas like '%Sel%'
""")

display(result)

In [0]:
spark_df2.createOrReplaceTempView("df2")
result = spark.sql("""
  SELECT
    *
  FROM df2 
""")

result.write.mode("overwrite").saveAsTable("dbacademy.default.my_earthquake_alert")

In [0]:
%sql
select * from dbacademy.default.my_earthquake_alert

In [0]:
%sql
select * from _sqldf